## Skydio X10 / VT300-L Thermal Orthomosaic Pipeline
===================================================
Uses SeaDroneLib (MosaicSeadron) to georeference and mosaic raw 16-bit
thermal TIFFs extracted from Skydio R-JPEGs.

This script fixes the three bugs found in skydio_batch_setup.py:

  Bug #1 — compute_flight_lines() was called with per-image camera-Yaw
            values, which are meaningless for line detection because the
            Skydio gimbal rotates freely. The result was only 2 bogus
            "lines" instead of 15 real transects.
            FIX: use load_flight_lines() which reads the manually curated
            summary.yml you already created.

  Bug #2 — georefence_images() was called (processes JPEGs from main/).
            Thermal mosaic needs georefence_bands() which processes the
            16-bit TIFFs from bands/.
            FIX: call georefence_bands() with a uint16 profile.

  Bug #3 — Because of Bugs 1 & 2, only 2 images were georeferenced.
            The merge step received degenerate geometry and crashed with
            "ValueError: cannot convert float NaN to integer".
            FIX: fixing Bugs 1 & 2 gives merge() the full set of rasters
            it needs to compute a valid bounding box.

Prerequisites (Phase 2 environment — Python 3.10, seadronelib-venv active)
---------------------------------------------------------------------------
  source seadronelib-venv/Scripts/activate   # Git Bash
  jupyter notebook                            # then open this as a .py or
                                              # paste cells into dji.ipynb

Directory layout expected
--------------------------
  D:/Thermal_Mosaic/
      main/        ← Skydio R-JPEGs  (S1007773_R.JPG … S1008289_R.JPG)
      bands/       ← 16-bit TIFFs produced by extract_thermal.py
                      (S1007773_R.tif … S1008289_R.tif)
      metadata.csv ← produced by the ExifTool cell in skydio_batch_setup.py
      summary.yml  ← your manually edited flight-line file (summary_mannual.yml)

Usage
-----
  Run top-to-bottom.  Outputs land in:
      D:/Thermal_Mosaic/georeferences/bands/   ← one warped TIFF per image
      D:/Thermal_Mosaic/merges/bands/          ← final thermal orthomosaic

In [3]:

# ============================================================
# 0.  IMPORTS
# ============================================================
import os
import glob
import rasterio
import pandas as pd
from dataclasses import asdict

from seadrone.data_structures import (
    Profile,
    Partition,
    GeoreferencePartition,
    MergePartition,
)
from seadrone.processing import FlightProcessor, Sensor, FlightMode


In [4]:


# ============================================================
# 1.  PATHS  — edit only this block
# ============================================================
PROJECT_PATH        = r"D:\Thermal_Mosaic"
MAIN_FOLDER         = os.path.join(PROJECT_PATH, "main")    # R-JPEGs
BANDS_FOLDER        = os.path.join(PROJECT_PATH, "bands")   # 16-bit TIFFs
METADATA_CSV        = os.path.join(PROJECT_PATH, "metadata.csv")
FLIGHT_LINES_YML    = os.path.join(PROJECT_PATH, "summary.yml")  # rename from summary_mannual.yml

GEOREFERENCE_OUT    = os.path.join(PROJECT_PATH, "georeferences", "bands")
MERGE_OUT           = os.path.join(PROJECT_PATH, "merges", "bands")

os.makedirs(GEOREFERENCE_OUT, exist_ok=True)
os.makedirs(MERGE_OUT, exist_ok=True)


# ============================================================
# 2.  SENSOR DEFINITION  (Skydio VT300-L physical specs)
# ============================================================
# Pixel pitch = 12 µm  →  sensor_x = 640 × 0.012 mm = 7.68 mm
#                          sensor_y = 512 × 0.012 mm = 6.14 mm
SKYDIO_SENSOR = Sensor(
    name         = "Skydio_VT300-L",
    width        = 640,
    height       = 512,
    focal_length = 13.6,    # mm  (from EXIF:FocalLength)
    sensor_x     = 7.68,    # mm
    sensor_y     = 6.14,    # mm
    bands_number = 1,
    band_names   = ["thermal"],
)


In [ ]:


# ============================================================
# 3.  RASTER PROFILES
# ============================================================
# Thermal TIFFs from extract_thermal.py are 16-bit single-band.
BANDS_PROFILE = Profile(
    dtype  = "uint16",   # ← must match the actual TIFF dtype
    count  = 1,          # single thermal band
    height = SKYDIO_SENSOR.height,
    width  = SKYDIO_SENSOR.width,
    nodata = 0,
)

# If you also want to georeference the colour-palette JPEGs for reference:
FLIGHT_PROFILE = Profile(
    dtype  = "uint8",
    count  = 3,          # RGB JPEG
    height = SKYDIO_SENSOR.height,
    width  = SKYDIO_SENSOR.width,
    nodata = 0,
)



In [5]:

# ============================================================
# 4.  LOAD METADATA  (already extracted by ExifTool step)
# ============================================================
print("Loading metadata …")
processor = FlightProcessor()

flight_metadata = processor.load_metadata(
    os.path.dirname(METADATA_CSV),
    os.path.basename(METADATA_CSV),
)
print(f"  Loaded {len(flight_metadata)} image records")

# ── Inject Skydio sensor columns ──────────────────────────────────────────
# SeaDroneLib looks for these exact column names when use_metadata=True.
flight_metadata["FocalLength"]   = SKYDIO_SENSOR.focal_length
flight_metadata["ImageWidth"]    = SKYDIO_SENSOR.width
flight_metadata["ImageHeight"]   = SKYDIO_SENSOR.height
flight_metadata["SensorX"]       = SKYDIO_SENSOR.sensor_x
flight_metadata["SensorY"]       = SKYDIO_SENSOR.sensor_y

# Duplicate GPS/gimbal columns to the names SeaDroneLib expects internally.
flight_metadata["GPSLatitude"]      = flight_metadata["Latitude"]
flight_metadata["GPSLongitude"]     = flight_metadata["Longitude"]
flight_metadata["GPSAltitude"]      = flight_metadata["Altitude"]
flight_metadata["GimbalYawDegree"]  = flight_metadata["Yaw"]
flight_metadata["GimbalPitchDegree"]= flight_metadata["Pitch"]
flight_metadata["GimbalRollDegree"] = flight_metadata["Roll"]

# ── ID column ─────────────────────────────────────────────────────────────
# georefence_bands() uses metadata["ID"] to find each TIFF in bands/.
# The TIFF names produced by extract_thermal.py are  S1007773_R.tif  etc.
flight_metadata["ID"] = (
    flight_metadata["Source"]
    .str.replace(r"\.JPG$", ".tif", case=False, regex=True)
)

# Sanity check
required_cols = ["Latitude", "Longitude", "Altitude",
                 "Yaw", "Pitch", "Roll",
                 "FocalLength", "SensorX", "SensorY",
                 "ImageWidth", "ImageHeight"]
nulls = flight_metadata[required_cols].isnull().sum()
if nulls.any():
    print("WARNING — null values found:\n", nulls[nulls > 0])
else:
    print("  All required columns populated, no nulls ✓")


Loading metadata …
  Loaded 173 image records
  All required columns populated, no nulls ✓


In [6]:


# ============================================================
# 5.  LOAD FLIGHT LINES FROM summary.yml
#
#     ⚠ THIS IS THE CRITICAL FIX ⚠
#     The original script called compute_flight_lines(), which tries to
#     infer transect boundaries from the Yaw column.  On the Skydio X10
#     the gimbal rotates freely and independently of the vehicle heading,
#     so per-image Yaw is essentially random noise — the auto-detector
#     found only 2 "lines" from 173 images.
#
#     load_flight_lines() reads your hand-curated summary.yml which
#     explicitly names the start and end image for each of the 15
#     transects.  It converts filename spans to integer row-index ranges
#     in the metadata DataFrame so georefence_bands() knows exactly which
#     subset of rows belongs to each transect.
# ============================================================
print("\nLoading flight lines from summary.yml …")
flight_lines = processor.load_flight_lines(
    os.path.dirname(FLIGHT_LINES_YML),
    os.path.basename(FLIGHT_LINES_YML),
)
print(f"  Loaded {len(flight_lines)} flight lines ✓")
for i, fl in enumerate(flight_lines):
    print(f"    Line {i+1}: rows {fl['start']} → {fl['end']}")


# ============================================================
# 6.  PARTITIONS
# ============================================================
partitions = {
    "all":    Partition("all",    0, None, 1),
    "even":   Partition("even",   0, None, 2),
    "odd":    Partition("odd",    1, None, 2),
}
PARTITIONS_TO_GEOREFERENCE = ["all"]
PARTITIONS_TO_MERGE        = ["all"]

# Flip axis=1 corrects the vertical orientation of the Skydio thermal sensor.
FLIP_AXIS = [1]


# ============================================================
# 7.  GEOREFERENCE THERMAL TIFFs
#
#     georefence_bands() is the correct function for raster bands (TIFFs).
#     georefence_images() is for JPEG visual images — it reads colour
#     space info from the JPEG that a 16-bit single-band TIFF does not
#     have, causing it to silently produce wrong output or crash.
# ============================================================
print("\nGeoreferencing thermal TIFFs …")

for partition_key in PARTITIONS_TO_GEOREFERENCE:
    part = partitions[partition_key]
    geo_partition = GeoreferencePartition(
        name    = part.name,
        start   = part.start,
        end     = part.end,
        steps   = part.steps,
        profile = BANDS_PROFILE,   # ← uint16, count=1
    )

    processor.georefence_bands(
        metadata      = flight_metadata,
        partition     = geo_partition,
        flight_lines  = flight_lines,
        in_folder     = BANDS_FOLDER,       # ← bands/  not main/
        out_folder    = GEOREFERENCE_OUT,
        flip_axis     = FLIP_AXIS,
        use_metadata  = True,
        overwrite     = True,
    )

georef_tifs = glob.glob(os.path.join(GEOREFERENCE_OUT, "**", "*.tif"), recursive=True)
print(f"  Georeferenced {len(georef_tifs)} TIFFs ✓")




Loading flight lines from summary.yml …
  Loaded 0 flight lines ✓

Georeferencing thermal TIFFs …


NameError: name 'BANDS_PROFILE' is not defined

In [ ]:

# ============================================================
# 8.  MERGE INTO ORTHOMOSAIC
#
#     Now that all 15 transects are georeferenced the merge step has a
#     valid bounding box and the NaN → int crash cannot occur.
# ============================================================
print("\nMerging into thermal orthomosaic …")

for partition_key in PARTITIONS_TO_MERGE:
    part = partitions[partition_key]
    merge_partition = MergePartition(
        name  = part.name,
        start = part.start,
        end   = part.end,
        steps = part.steps,
        skip  = 1,   # use every frame (you already excluded turns in summary.yml)
    )

    processor.merge(
        metadata      = flight_metadata,
        partition     = merge_partition,
        flight_lines  = flight_lines,
        in_folder     = GEOREFERENCE_OUT,
        out_folder    = MERGE_OUT,
        method        = "mean",   # 'mean' blends overlapping footprints smoothly
        band_names    = None,
    )

mosaic_files = glob.glob(os.path.join(MERGE_OUT, "**", "*.tif"), recursive=True)
print(f"\n{'='*60}")
print(f"Done!  Orthomosaic(s) written to: {MERGE_OUT}")
for f in mosaic_files:
    print(f"  {os.path.basename(f)}")
print(f"{'='*60}")


# ============================================================
# 9.  OPTIONAL — quick quality check on the output mosaic
# ============================================================
if mosaic_files:
    with rasterio.open(mosaic_files[0]) as src:
        print(f"\nMosaic properties:")
        print(f"  CRS    : {src.crs}")
        print(f"  Shape  : {src.height} rows × {src.width} cols")
        print(f"  Bands  : {src.count}")
        print(f"  Dtype  : {src.dtypes}")
        print(f"  Bounds : {src.bounds}")
        data = src.read(1)
        valid = data[data != 0]
        if valid.size:
            print(f"  Min/Max raw DN : {valid.min()} / {valid.max()}")
        else:
            print("  WARNING: all pixels are nodata (0). Check flip_axis and sensor specs.")